# HomePlatform competitor audit validation

**Archived supporting evidence; non-authoritative.** Reproducible, offline structural and data-quality checks for the audit with a research cutoff of **2026-08-29**. Outputs were cleared after archival. This notebook validates the archived artifact set; it does not fetch network data or modify HomePlatform source code.

In [ ]:
from collections import Counter
from pathlib import Path
from urllib.parse import unquote
import csv
import hashlib
import json
import re

cwd = Path.cwd().resolve()
candidate = cwd / "docs/research/archive/competitor-audit-2026-08-29"
ROOT = cwd if cwd.name == "competitor-audit-2026-08-29" else candidate
DOCS_ROOT = next(parent for parent in ROOT.parents if parent.name == "docs")
ARCH_ROOT = DOCS_ROOT / "architecture/research"
assert ROOT.name == "competitor-audit-2026-08-29" and ROOT.exists(), ROOT
print({"artifact_root": str(ROOT), "research_cutoff": "2026-08-29"})

In [ ]:
matrix_path = ROOT / "COMPETITOR-FEATURE-MATRIX.csv"
with matrix_path.open(newline="", encoding="utf-8") as handle:
    matrix = list(csv.DictReader(handle))

required_columns = [
    "product", "segment", "market_status", "category", "feature",
    "support", "qualification", "source_ids",
]
assert list(matrix[0]) == required_columns
assert len(matrix) == 1_875, f"expected 1,875 rows, got {len(matrix)}"

expected_products = {
    "Cozi", "FamilyWall", "Family Tools", "Jam Family Calendar", "TimeTree",
    "Any.do Family", "Ohai.ai", "Sweepy", "Skylight Calendar",
    "OurFamilyWizard", "AppClose", "AnyList", "Splitwise", "Dwellin",
    "Google Family",
}
products = {row["product"] for row in matrix}
dimensions = {(row["category"], row["feature"]) for row in matrix}
keys = [(row["product"], row["category"], row["feature"]) for row in matrix]
allowed = {"Y", "P", "X", "U"}

assert products == expected_products, products ^ expected_products
assert len(dimensions) == 125, len(dimensions)
assert len({row["category"] for row in matrix}) == 13
assert len(keys) == len(set(keys)), "duplicate product/category/feature row"
assert {row["support"] for row in matrix} <= allowed
assert all(row["qualification"].strip() for row in matrix)
assert all(re.fullmatch(r"S\d{3}(;S\d{3})*", row["source_ids"]) for row in matrix)

rows_per_product = Counter(row["product"] for row in matrix)
assert set(rows_per_product.values()) == {125}, rows_per_product
print({
    "deep_dive_products": len(products),
    "feature_dimensions": len(dimensions),
    "matrix_rows": len(matrix),
    "categories": len({row["category"] for row in matrix}),
    "support_counts": dict(sorted(Counter(row["support"] for row in matrix).items())),
})

In [ ]:
pain_path = ROOT / "USER-PAIN-MATRIX.csv"
with pain_path.open(newline="", encoding="utf-8") as handle:
    pains = list(csv.DictReader(handle))

assert len(pains) >= 12
assert len({row["pain_id"] for row in pains}) == len(pains)
assert all(re.fullmatch(r"P\d{2}", row["pain_id"]) for row in pains)
assert all(row["frequency_signal"].strip() for row in pains)
assert all(row["limitations"].strip() and row["source_ids"].strip() for row in pains)
assert all(re.fullmatch(r"S\d{3}(;S\d{3})*", row["source_ids"]) for row in pains)
print({
    "pain_rows": len(pains),
    "frequency_signals": sorted({row["frequency_signal"] for row in pains}),
})

In [ ]:
sources_text = (ROOT / "SOURCES.md").read_text(encoding="utf-8")
inventory_numbers = [
    int(value)
    for value in re.findall(r"^\|\s*(\d+)\s*\|", sources_text, flags=re.M)
    if 1 <= int(value) <= 46
]
assert sorted(set(inventory_numbers)) == list(range(1, 47)), sorted(set(inventory_numbers))
assert len(inventory_numbers) == 46, "inventory numbering is duplicated"

defined_source_ids = set(
    re.findall(r"^\|\s*(S\d{3})\s*\|", sources_text, flags=re.M)
)
all_text = "\n".join(
    path.read_text(encoding="utf-8")
    for path in sorted(DOCS_ROOT.rglob("*"))
    if path.suffix.lower() in {".md", ".csv"} and path != ROOT / "SOURCES.md"
)
used_source_ids = set(re.findall(r"\bS\d{3}\b", all_text))
undefined_source_ids = sorted(used_source_ids - defined_source_ids)
unused_source_ids = sorted(defined_source_ids - used_source_ids)
assert not undefined_source_ids, undefined_source_ids
assert not unused_source_ids, unused_source_ids
print({
    "market_inventory_products": len(set(inventory_numbers)),
    "defined_source_ids": len(defined_source_ids),
    "used_source_ids": len(used_source_ids),
    "undefined_source_ids": undefined_source_ids,
    "unused_source_ids": unused_source_ids,
})

In [ ]:
report = (ROOT / "HOMEPLATFORM-COMPETITOR-AUDIT.md").read_text(encoding="utf-8")
required_sections = [
    "## Executive Summary",
    "## Market Landscape",
    "## Competitor Shortlist",
    "## Feature Matrix",
    "## Workflow Comparison",
    "## User Pain",
    "## Table Stakes",
    "## HomePlatform Feature Gaps",
    "## Household / Identity Model Findings",
    "## Bounded Context Findings",
    "## Aggregate Findings",
    "## Architecture Pressure Test",
    "## Competitor Architecture Evidence",
    "## Security & Privacy",
    "## Realtime Requirements",
    "## Recurrence / Scheduling Requirements",
    "## Notification Requirements",
    "## Monetization",
    "## Onboarding",
    "## Product Opportunities",
    "## Six-Month Roadmap Impact",
    "## Portfolio DDD Review",
    "## Things NOT To Build",
    "## Evidence Limitations",
    "## Final Recommendations",
]
missing_sections = [section for section in required_sections if section not in report]
assert not missing_sections, missing_sections

decision_checks = {
    "status_complete": r"STATUS:\s+COMPLETE",
    "modular_monolith_keep": r"CURRENT MODULAR MONOLITH:\s+KEEP",
    "domain_major_rework": r"CURRENT DOMAIN MODEL:\s+MAJOR REWORK",
    "roadmap_modify": r"CURRENT SIX-MONTH ROADMAP:\s+MODIFY",
    "source_unchanged": r"SOURCE CODE MODIFIED:\s+NO",
    "next_domain_question": r"NEXT DOMAIN QUESTION TO RESOLVE:\s+\S",
    "next_file": r"NEXT FILE TO CREATE AFTER THIS RESEARCH:\s+`[^`]+`",
}
failed_decisions = [name for name, pattern in decision_checks.items() if not re.search(pattern, report)]
assert not failed_decisions, failed_decisions

final_block = report[report.rfind("HOMEPLATFORM-COMPETITOR-PRODUCT-ARCHITECTURE-AUDIT-01"):]
exact_values = {
    "STATUS": "COMPLETE",
    "PRODUCT COVERAGE SCORE": "2/10",
    "DDD ARCHITECTURE FIT": "6/10",
    "PORTFOLIO QUALITY": "4.5/10",
    "CURRENT DOMAIN MODEL": "MAJOR REWORK",
    "CURRENT MODULAR MONOLITH": "KEEP",
    "CURRENT SIX-MONTH ROADMAP": "MODIFY",
    "SOURCE CODE MODIFIED": "NO",
}
for label, expected in exact_values.items():
    match = re.search(rf"^{re.escape(label)}:\s*$\n^([^\n]+)$", final_block, flags=re.M)
    assert match and match.group(1).strip() == expected, (label, match.group(1).strip() if match else None)

list_specs = [
    ("TOP 10 FEATURES HOMEPLATFORM SHOULD HAVE", "TOP 5 OVERLOOKED DOMAIN CONCEPTS", 10),
    ("TOP 5 OVERLOOKED DOMAIN CONCEPTS", "TOP 5 ARCHITECTURAL PRESSURES", 5),
    ("TOP 5 ARCHITECTURAL PRESSURES", "TOP 5 FEATURES TO NOT BUILD YET", 5),
    ("TOP 5 FEATURES TO NOT BUILD YET", "CURRENT DOMAIN MODEL", 5),
]
for start, end, expected_count in list_specs:
    segment = final_block.split(start + ":", 1)[1].split(end + ":", 1)[0]
    assert len(re.findall(r"^\d+\.\s", segment, flags=re.M)) == expected_count, start

profiles = (ROOT / "COMPETITOR-PROFILES.md").read_text(encoding="utf-8")
numbered_profiles = re.findall(r"^##\s+(\d+)\.\s+(.+)$", profiles, flags=re.M)
assert [int(number) for number, _ in numbered_profiles] == list(range(1, 16))
assert {name for _, name in numbered_profiles} == expected_products
assert "NOT ANALYSED" not in all_text
print({
    "required_report_sections": len(required_sections),
    "deep_dive_profiles": len(numbered_profiles),
    "final_decision_checks": len(decision_checks),
    "exact_final_values": len(exact_values),
    "exact_final_list_counts": [spec[2] for spec in list_specs],
})

In [ ]:
required_files = [
    ROOT / "HOMEPLATFORM-COMPETITOR-AUDIT.md",
    ROOT / "COMPETITOR-FEATURE-MATRIX.csv",
    ROOT / "COMPETITOR-PROFILES.md",
    ROOT / "USER-PAIN-MATRIX.csv",
    ROOT / "ARCHITECTURE-EVIDENCE.md",
    ROOT / "FEATURE-GAP-ANALYSIS.md",
    ROOT / "PRODUCT-OPPORTUNITY-MAP.md",
    ROOT / "SOURCES.md",
    ARCH_ROOT / "COMPETITOR-IMPACT-ON-DDD.md",
    ROOT / "PROPOSED-BOUNDED-CONTEXTS.md",
    ARCH_ROOT / "AGGREGATE-PRESSURE-TEST.md",
    ROOT / "ROADMAP-CHANGE-PROPOSAL.md",
]
missing_files = [str(path) for path in required_files if not path.is_file()]
assert not missing_files, missing_files

local_link_pattern = re.compile(r"\[[^\]]+\]\(([^)]+)\)")
broken_links = []
for markdown_file in sorted(DOCS_ROOT.rglob("*.md")):
    markdown = markdown_file.read_text(encoding="utf-8")
    for raw_target in local_link_pattern.findall(markdown):
        target = raw_target.strip().split()[0].strip("<>")
        if target.startswith(("http://", "https://", "mailto:", "#")):
            continue
        file_part = unquote(target.split("#", 1)[0])
        if file_part and not (markdown_file.parent / file_part).resolve().exists():
            broken_links.append((str(markdown_file.relative_to(DOCS_ROOT)), target))
assert not broken_links, broken_links

manifest = {
    str(path.relative_to(DOCS_ROOT)): {
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }
    for path in required_files
}
print(json.dumps({
    "required_deliverables": len(required_files),
    "missing_deliverables": len(missing_files),
    "broken_relative_links": len(broken_links),
    "manifest_entries": len(manifest),
}, indent=2))

In [ ]:
validation_summary = {
    "status": "PASS",
    "research_cutoff": "2026-08-29",
    "inventory_products": 46,
    "deep_dive_products": len(products),
    "feature_dimensions": len(dimensions),
    "feature_matrix_rows": len(matrix),
    "pain_rows": len(pains),
    "required_deliverables": len(required_files),
    "undefined_source_ids": len(undefined_source_ids),
    "unused_source_ids": len(unused_source_ids),
    "broken_relative_links": len(broken_links),
}
print(json.dumps(validation_summary, indent=2))